# Deep Learning 080 — The Transformer Encoder

Companion notebook to the lesson. Every component is now built, so this notebook assembles
the encoder — and then tests the two design choices whose standard explanations do not
survive contact with a measurement.

| Claim | What we measure |
|---|---|
| the shape stays $n \times 512$ | asserted at every stage, and after six blocks |
| attention is where the parameters are | **false** — the feed-forward net is **2.00×** attention |
| the encoder vs the vocabulary | 18,914,304 against 18,944,000 — **0.16%** apart |
| "self-attention is linear" | **false** — linearity test gives **1.597** |
| so why the feed-forward net? | pure attention **collapses every token by depth 2** |
| ...and what stops the collapse | not the feed-forward net (buys 1 layer) — the **residual** |
| "residuals stop vanishing gradients" | **not here** — with Add & Norm the ratio is **1.04×** |

Needs `torch` (CPU is fine). No training and no GPU.

In [ ]:
import math
import torch
import torch.nn as nn

D, D_FF, H = 512, 2048, 8

class MultiHeadAttention(nn.Module):
    def __init__(self, d=D, h=H):
        super().__init__()
        self.h, self.dk = h, d // h
        self.wq, self.wk = nn.Linear(d, d), nn.Linear(d, d)
        self.wv, self.wo = nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, x):
        n, d = x.shape
        split = lambda t: t.view(n, self.h, self.dk).transpose(0, 1)
        q, k, v = split(self.wq(x)), split(self.wk(x)), split(self.wv(x))
        w = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(self.dk), dim=-1)
        return self.wo((w @ v).transpose(0, 1).reshape(n, d))

class EncoderBlock(nn.Module):
    '''attention -> Add & Norm -> feed-forward -> Add & Norm.  That is the whole block.'''
    def __init__(self, residual=True, pre_ln=False):
        super().__init__()
        self.attn = MultiHeadAttention()
        self.ff = nn.Sequential(nn.Linear(D, D_FF), nn.ReLU(), nn.Linear(D_FF, D))
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.residual, self.pre_ln = residual, pre_ln

    def forward(self, x):
        if self.pre_ln:
            x = x + self.attn(self.ln1(x)) if self.residual else self.attn(self.ln1(x))
            x = x + self.ff(self.ln2(x))   if self.residual else self.ff(self.ln2(x))
            return x
        x = self.ln1(x + self.attn(x)) if self.residual else self.ln1(self.attn(x))
        x = self.ln2(x + self.ff(x))   if self.residual else self.ln2(self.ff(x))
        return x

print(EncoderBlock())

That is the entire encoder block. The `Nx` in the famous diagram means six of these stacked,
architecturally identical and each with **its own** parameters.

## Part A — Where the parameters actually are

Nine lessons on attention leave a strong impression that attention is where the model lives.
Derive the count on paper first, then check it against the module.

In [ ]:
attn = 4 * (D * D + D)                          # W_q, W_k, W_v, W_o, with biases
ff   = (D * D_FF + D_FF) + (D_FF * D + D)       # 512 -> 2048 -> 512
lns  = 2 * (2 * D)                              # two layer norms, gamma and beta
block = attn + ff + lns

torch.manual_seed(0)
counted = sum(p.numel() for p in EncoderBlock().parameters())
assert counted == block, (block, counted)

print(f"multi-head attention : {attn:>10,}   {100*attn/block:5.1f}%")
print(f"feed-forward network : {ff:>10,}   {100*ff/block:5.1f}%")
print(f"two layer norms      : {lns:>10,}   {100*lns/block:5.1f}%")
print(f"one encoder block    : {block:>10,}   (module agrees: {counted:,})")
print(f"\nfeed-forward / attention = {ff/attn:.2f}x")

In [ ]:
six = 6 * block
embed = 37_000 * D                              # the 2017 model's shared vocabulary
print(f"six encoder blocks   : {six:,}")
print(f"embedding table      : {embed:,}")
print(f"difference           : {abs(six-embed):,}  =  {100*abs(six-embed)/embed:.2f}%")

Two thirds of every block is the feed-forward network — **exactly twice** the attention,
because attention holds four $512\times512$ matrices while the feed-forward network holds two
$512\times2048$ ones.

And the six-block encoder lands within **0.16%** of the embedding table. Six architectures in
this course had the vocabulary-facing layer dominating the parameter count; this is the
crossing point, and everything since has been on the far side of it.

## Part B — The shape never changes, and that is the point

In [ ]:
torch.manual_seed(80)
n = 3                                            # "How are you"
x = torch.randn(n, D)                            # already embedded + positionally encoded
blk = EncoderBlock()

a  = blk.attn(x);        print(f"multi-head attention out {tuple(a.shape)}")
r1 = blk.ln1(x + a);     print(f"Add & Norm               {tuple(r1.shape)}")
f  = blk.ff(r1);         print(f"feed-forward out         {tuple(f.shape)}")
r2 = blk.ln2(r1 + f);    print(f"Add & Norm               {tuple(r2.shape)}")

y = x
for _ in range(6):
    y = blk(y)
print(f"after six blocks         {tuple(y.shape)}")
assert all(tuple(t.shape) == (n, D) for t in (a, r1, f, r2, y))

Shape invariance is what makes "stack six of them" a legal sentence, and it is a constraint
that fixes several other choices: why multi-head attention concatenates its eight
64-dimensional heads back to 512, why $W_o$ is $512\times512$, and why the feed-forward network
expands to 2048 and projects **straight back down**. The 2048 is never visible outside the
feed-forward network. The residual also requires it: $x + \text{sublayer}(x)$ is only defined
if the sublayer preserves the shape.

One detail the diagram hides, and it matters for the rest of the notebook.

In [ ]:
together = blk.ff(r1)
one_by_one = torch.cat([blk.ff(r1[i:i+1]) for i in range(n)], dim=0)
print(f"max |ff(all rows) - ff(row by row)| = {(together-one_by_one).abs().max():.2e}")

The feed-forward network is **position-wise**: the same network applied to each token
independently, with no mixing across positions. Layer norm is per-token too (lesson 079: it
never reads outside the row), and the residual addition is elementwise.

**So multi-head attention is the only operation in the entire block where tokens meet.** A
block is one round of conversation followed by everyone going away and thinking alone.

## Part C — "Self-attention is linear" — it is not

Ask why the feed-forward network is there and you will be told: attention is linear, so a ReLU
is needed or the stack collapses into one linear map. It is a satisfying answer with a clean
precedent. Test it.

In [ ]:
torch.manual_seed(3080)
f = MultiHeadAttention()
a, b = 2.0, -3.0
x, y = torch.randn(6, D), torch.randn(6, D)

lhs = f(a * x + b * y)
rhs = a * f(x) + b * f(y)
print(f"relative error = {((lhs - rhs).norm() / rhs.norm()).item():.3f}   (linear would be 0)")

In [ ]:
# where the non-linearity lives: the softmax
print(f"{'input scale':>12} {'largest weight':>16} {'row entropy':>13}")
for s in (0.25, 1.0, 4.0, 16.0):
    q = f.wq(s * x).view(6, H, -1).transpose(0, 1)
    k = f.wk(s * x).view(6, H, -1).transpose(0, 1)
    w = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(f.dk), dim=-1)
    ent = -(w * (w + 1e-12).log()).sum(-1).mean()
    print(f"{s:>12.2f} {w.max().item():>16.4f} {ent.item():>13.4f}")
print(f"\n(uniform over 6 tokens would be {math.log(6):.4f})")

Multiplying the input by 16 turns a nearly uniform attention distribution into a hard
selection of one token. **No linear map does that.** It is the same phenomenon lesson 074 met
from the other side when it divided by $\sqrt{d_k}$ to stop large dot products saturating the
softmax.

So the popular justification for the feed-forward network answers a problem that does not
exist — which leaves the question genuinely open.

## Part D — What a stack of pure attention actually does

Strip the block to attention alone, stack it, and watch the token vectors. The quantity to
watch is how far the token matrix is from having every row identical.

In [ ]:
def spread(X):
    '''0 means total collapse: every token has become the same vector.'''
    return ((X - X.mean(0, keepdim=True)).norm() / X.norm()).item()

def mean_cosine(X):
    Z = X / X.norm(dim=1, keepdim=True)
    n = len(X)
    return ((Z @ Z.T).sum().item() - n) / (n * (n - 1))

DEPTHS = [0, 1, 2, 3, 4, 6, 12, 24, 32]

variants = {
    "attention only": lambda b, x: b.attn(x),
    "+ feed-forward": lambda b, x: b.ln2(b.ff(b.ln1(b.attn(x)))),
    "full block":     lambda b, x: b(x),
}

torch.manual_seed(4080)
x0 = torch.randn(16, D)                    # drawn first, matching the lesson script

for name, step in variants.items():
    torch.manual_seed(4080)
    blocks = [EncoderBlock() for _ in range(max(DEPTHS))]
    x = x0.clone()
    print(f"\n{name}")
    print(f"  {'depth':>6} {'||res||/||X||':>15} {'mean cosine':>13}")
    with torch.no_grad():
        for depth in range(max(DEPTHS) + 1):
            if depth in DEPTHS:
                print(f"  {depth:>6} {spread(x):>15.2e} {mean_cosine(x):>13.4f}")
            if depth < max(DEPTHS):
                x = step(blocks[depth], x)

**Pure attention drives every token onto the same vector, immediately.** One block takes the
mean cosine from 0.004 to 0.89; by depth 2 the residual is `2e-05`; by depth 3 it is float32
zero. That fall is far faster than exponential — the signature Dong, Cordonnier and Loukas
(2021) proved when they showed pure attention loses rank *doubly* exponentially in depth.

A network whose tokens are all equal cannot represent a sentence, so **this is the failure the
block exists to prevent**. But look at where the credit goes: the feed-forward network buys
exactly **one layer** (collapse at depth 3 instead of 2). The **residual connection** is what
actually prevents it — six blocks still leave the tokens at cosine 0.16, and thirty-two leave
them at 0.88.

## Part E — And the textbook reason for residuals is wrong too

Residual connections are usually justified by vanishing gradients, borrowing the argument from
ResNet. The architecture has Add & Norm, so the claim is directly testable.

In [ ]:
def grad_at_input(depth, residual, with_ln):
    torch.manual_seed(5080)
    blocks = nn.ModuleList([EncoderBlock(residual=residual) for _ in range(depth)])
    if not with_ln:
        for b in blocks:
            b.ln1, b.ln2 = nn.Identity(), nn.Identity()
    x = torch.randn(16, D, requires_grad=True)
    target = torch.randn(16, D)
    h = x
    for b in blocks:
        h = b(h)
    ((h - target) ** 2).mean().backward()
    return x.grad.norm().item()

print("Add & Norm intact (the real architecture):")
print(f"{'blocks':>7} {'with residual':>16} {'without':>14} {'ratio':>10}")
for depth in (6, 12, 24):
    a, b = grad_at_input(depth, True, True), grad_at_input(depth, False, True)
    print(f"{depth:>7} {a:>16.4e} {b:>14.4e} {a/b:>9.2f}x")

**A negative result.** At the depth the paper actually uses, removing every residual connection
changes the gradient reaching the input by about **4%**. There is no vanishing gradient here
to rescue.

The explanation is lesson 079, Part G: layer norm's Jacobian scales as $1/\sigma$, so it is
already regulating the gradient's magnitude — the job residuals usually get credit for. Take
the layer norms out and see what happens.

In [ ]:
print("Layer norms replaced by the identity:")
print(f"{'blocks':>7} {'with residual':>16} {'without':>14}")
for depth in (6, 12, 24):
    a, b = grad_at_input(depth, True, False), grad_at_input(depth, False, False)
    note = "  <- underflowed to zero" if b == 0.0 else f"  ratio {a/b:.1e}x"
    print(f"{depth:>7} {a:>16.4e} {b:>14.4e}{note}")

Without layer norm, a 24-block stack without residuals delivers a gradient that has
**underflowed to exactly zero** — the vanishing gradient in its purest form. But the real
architecture *has* layer norm, and with it the residual is worth 4%.

**So the two components are not redundant, and the standard explanations have their jobs
swapped: layer norm handles the gradient scale, and the residual connection handles the
collapse.** Each does something the other cannot.

## Part F — Where the norm goes

The paper writes Add & Norm as $\text{LN}(x + \text{sublayer}(x))$ — **post-LN**. Essentially
every model since GPT-2 uses $x + \text{sublayer}(\text{LN}(x))$ — **pre-LN** — leaving the
residual path uninterrupted.

In [ ]:
print(f"{'depth':>6} {'post-LN':>12} {'pre-LN':>12}   (gradient on block 1's parameters)")
for depth in (6, 12, 24, 48):
    row = []
    for pre in (False, True):
        torch.manual_seed(6080)
        blocks = nn.ModuleList([EncoderBlock(pre_ln=pre) for _ in range(depth)])
        h, target = torch.randn(16, D), torch.randn(16, D)
        for b in blocks:
            h = b(h)
        ((h - target) ** 2).mean().backward()
        row.append(torch.cat([p.grad.reshape(-1) for p in blocks[0].parameters()]).norm().item())
    print(f"{depth:>6} {row[0]:>12.4f} {row[1]:>12.4f}")

**Measured:** post-LN's gradient is remarkably flat — about 0.5 at every depth from 6 to 48 —
while pre-LN's grows with depth, because its residual stream is never renormalised so
activations accumulate.

**Not shown, and worth being careful about:** the usual reason given for preferring pre-LN is
that post-LN requires a learning-rate **warmup** schedule, attributed to badly scaled gradients
at initialisation (Xiong et al., 2020). This measurement does not reproduce that — by this
probe post-LN is the better-behaved of the two at step zero. The warmup requirement is
**reported** in the literature and concerns training dynamics over many steps, which a single
forward and backward pass cannot settle. The 2017 model did use warmup; that much is in the
paper.

## Part G — The whole encoder, end to end

In [ ]:
torch.manual_seed(2026)
VOCAB = 1000

def positional_encoding(n_pos, d, base=10000.0):
    pos = torch.arange(n_pos).unsqueeze(1).float()
    i = torch.arange(d // 2).unsqueeze(0).float()
    ang = pos / base ** (2 * i / d)
    pe = torch.zeros(n_pos, d)
    pe[:, 0::2], pe[:, 1::2] = ang.sin(), ang.cos()
    return pe

embedding = nn.Embedding(VOCAB, D)
encoder = nn.ModuleList([EncoderBlock() for _ in range(6)])

tokens = torch.tensor([11, 42, 7])                    # "How are you"
x = embedding(tokens) * math.sqrt(D) + positional_encoding(len(tokens), D)
print(f"after the input block : {tuple(x.shape)}")

with torch.no_grad():
    for i, b in enumerate(encoder, 1):
        x = b(x)
        print(f"after encoder block {i} : {tuple(x.shape)}   mean cosine "
              f"{mean_cosine(x):.4f}")
print("\nthis is what the decoder will attend to (lesson 081).")

## What to take away

- The encoder is **six architecturally identical blocks with different parameters**, each doing
  multi-head attention → Add & Norm → feed-forward → Add & Norm, holding the shape at
  $n \times 512$ throughout.
- **Two thirds of every block is the feed-forward network** (2.00× attention), and the six
  blocks land within **0.16%** of the embedding table.
- **Attention is the only operation that mixes positions.**
- **"Attention is linear" is false** (relative error 1.597) — the softmax is the non-linearity.
- **Pure attention collapses every token onto the same vector by depth 2.** The feed-forward
  network delays it one layer; the **residual connection** prevents it.
- **Residuals do not rescue a vanishing gradient here** (1.04× at six blocks with Add & Norm).
  Layer norm handles the gradient; the residual handles the collapse.

**Exercises**

1. In Part D, keep the residual connection but delete the feed-forward network. Where does the
   collapse curve land relative to the three shown? What does that say about the credit split?
2. The lesson notes that Geva et al. (2021) read the feed-forward layers as key-value memories.
   Feed a fixed token through an untrained block and look at which of the 2048 hidden units
   fire. How sparse is it?
3. Set `pre_ln=True` throughout Part D. Does pre-LN change how fast the tokens collapse?
4. Scale $d_{ff}$ from 2048 down to 512 and up to 4096. How does the 0.16% dead heat with the
   embedding table move?